# When Japanese Font Settings Break

This notebook tests common situations where Matplotlib Japanese font settings disappear and shows which spell should be rerun: `import japanize_matplotlib`, `japanize_matplotlib.japanize()`, or figure recreation.

For clean results, this notebook runs most scenarios in isolated Python subprocesses. That makes each scenario close to a fresh Jupyter kernel.

In [ ]:
import os
import subprocess
import sys
import textwrap
from pathlib import Path


def run_case(name, code):
    result = subprocess.run(
        [sys.executable, "-c", textwrap.dedent(code)],
        cwd=Path.cwd(),
        env=os.environ.copy(),
        text=True,
        capture_output=True,
        check=False,
    )
    print(f"\n===== {name} =====")
    print("returncode:", result.returncode)
    if result.stdout:
        print("--- stdout ---")
        print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
    return result

In [ ]:
STATUS_CODE = r'''
import sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

def report(label):
    matches = [font for font in fm.fontManager.ttflist if font.name == "IPAexGothic"]
    print(f"[{label}]")
    print("japanize imported:", "japanize_matplotlib" in sys.modules)
    print("font.family:", plt.rcParams["font.family"])
    print("IPAexGothic in ttflist:", len(matches))
    try:
        print("findfont:", fm.findfont("IPAexGothic", fallback_to_default=False))
    except Exception as exc:
        print("findfont failed:", type(exc).__name__, exc)
    print()

def save_plot(path, title):
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.plot([1, 2, 3], [10, 20, 15], marker="o")
    ax.set_title(title)
    ax.set_xlabel("回数")
    ax.set_ylabel("値")
    fig.tight_layout()
    fig.savefig(path)
    print("saved:", path)
'''

print(STATUS_CODE[:300])

In [ ]:
run_case(
"fresh kernel without japanize_matplotlib",
STATUS_CODE + r'''
report("initial")
save_plot("/tmp/case01_without_japanize.png", "日本語 without japanize")
''',
)

In [ ]:
run_case(
"fresh kernel then import japanize_matplotlib",
STATUS_CODE + r'''
report("before import")
import japanize_matplotlib
report("after import")
save_plot("/tmp/case02_with_japanize.png", "日本語 with japanize")
''',
)

In [ ]:
run_case(
"plt.style.use resets rcParams; import again is not enough",
STATUS_CODE + r'''
import japanize_matplotlib
report("after import")

plt.style.use("default")
report("after style.use default")

import japanize_matplotlib
report("after second import")

japanize_matplotlib.japanize()
report("after japanize()")
save_plot("/tmp/case03_style_reapplied.png", "日本語 after japanize")
''',
)

In [ ]:
run_case(
"plt.rcdefaults resets font family",
STATUS_CODE + r'''
import japanize_matplotlib
report("after import")

plt.rcdefaults()
report("after rcdefaults")

japanize_matplotlib.japanize()
report("after japanize()")
save_plot("/tmp/case04_rcdefaults_reapplied.png", "日本語 after rcdefaults")
''',
)

In [ ]:
run_case(
"manual rcParams overwrite",
STATUS_CODE + r'''
import japanize_matplotlib
report("after import")

plt.rcParams["font.family"] = ["DejaVu Sans"]
report("after manual overwrite")

japanize_matplotlib.japanize()
report("after japanize()")
save_plot("/tmp/case05_manual_overwrite_reapplied.png", "日本語 after overwrite")
''',
)

In [ ]:
run_case(
"figure created before japanize_matplotlib",
STATUS_CODE + r'''
fig, ax = plt.subplots(figsize=(5, 3))
title = ax.set_title("日本語 before japanize")
ax.set_xlabel("回数")
ax.plot([1, 2, 3], [10, 20, 15])
print("title font before:", title.get_fontfamily())

import japanize_matplotlib
report("after import")
print("existing title font after import:", title.get_fontfamily())
fig.savefig("/tmp/case06_existing_text_before_fix.png")

title.set_fontfamily("IPAexGothic")
ax.xaxis.label.set_fontfamily("IPAexGothic")
fig.savefig("/tmp/case06_existing_text_after_manual_fix.png")
print("existing title font after manual fix:", title.get_fontfamily())

save_plot("/tmp/case06_new_figure_after_japanize.png", "日本語 new figure after japanize")
''',
)

In [ ]:
run_case(
"importlib.reload(package) is not enough; japanize() is clearer",
STATUS_CODE + r'''
import importlib
import japanize_matplotlib
report("after import")

plt.style.use("default")
report("after style.use default")

importlib.reload(japanize_matplotlib)
report("after importlib.reload(package)")

japanize_matplotlib.japanize()
report("after japanize()")
save_plot("/tmp/case07_reload_then_japanize.png", "日本語 after explicit japanize")
''',
)

## Colab-style bootstrap cell

Use this pattern near the top of a Colab notebook. If the runtime restarts, rerun this cell before any plotting cells.

In [ ]:
COLAB_BOOTSTRAP = r'''
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("japanize_matplotlib") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "japanize-matplotlib",
    ])

import japanize_matplotlib
japanize_matplotlib.japanize()
'''

print(COLAB_BOOTSTRAP)